# 05 · ClinicalBERT on the landmark data

Retrains the team's ClinicalBERT set-up (`ClinicalBERT_Train.ipynb`) on the **corrected, leak-free** splits from notebook 04
(`landmark_v2_{L}h/`). Same model (`emilyalsentzer/Bio_ClinicalBERT`), same learning rate, class-weighted loss.

Changes from the original, and why:

| Original | Here | Why |
|---|---|---|
| balanced 10k sample (50 % positive) | all positives + 3 negatives per positive | positives are now rare (≈5–10 %); keep every one |
| epoch chosen by val **AUROC** | chosen by val **AUPRC** | AUPRC is the honest metric at this imbalance |
| `MAX_LEN = 512` | 384 (truncation rate printed) | 99 % of texts fit (a first run at 256 cut off 63 % of texts — including the oxygen sentence at the end) |
| — | mixed precision, pre-tokenised | fits comfortably on a free Colab T4 |
| — | compared with, and averaged with, the structured model on the **same** test set | shows whether the text model adds anything |

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU*. About 20–25 minutes per landmark.
**Outputs:** `clinicalbert_landmark_results.json` (aggregate metrics only). The model itself is not saved unless you set `SAVE_MODEL = True`.

In [4]:
# ── Setup ─────────────────────────────────────────────────────
!pip -q install transformers

import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import rankdata

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

SAVE_DIR   = Path("/content/drive/MyDrive/USC/ICU-MM-landmark")   # where notebook 04 wrote landmark_v2_*h/
LANDMARKS  = [12]            # add 3, 6 if time allows: [3, 6, 12]
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN, BATCH_SIZE, EPOCHS, LR = 384, 16, 4, 2e-5
NEG_PER_POS = 3
SAVE_MODEL  = False
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU — switch the runtime to T4 GPU")
torch.manual_seed(SEED); np.random.seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | Tesla T4


In [5]:
def encode(df, shuffle=False, bs=BATCH_SIZE):
    enc = tokenizer(df["clinical_text"].tolist(), max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(df["respiratory_failure"].values, dtype=torch.long))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for ids, mask, _ in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def metrics(y, p):
    return {"AUROC": round(float(roc_auc_score(y, p)), 4), "AUPRC": round(float(average_precision_score(y, p)), 4),
            "AUPRC_chance": round(float(np.mean(y)), 4)}

def run(L):
    d = SAVE_DIR / f"landmark_v2_{L}h"
    train_df, val_df, test_df = (pd.read_csv(d / f"{s}_df.csv") for s in ("train", "val", "test"))
    print(f"\n=== Landmark {L}h === train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,} | "
          f"positive rate {train_df.respiratory_failure.mean()*100:.1f}%")

    # Token-length check
    lens = [len(x) for x in tokenizer(train_df["clinical_text"].sample(min(2000, len(train_df)), random_state=SEED).tolist())["input_ids"]]
    print(f"Tokens per text: median {int(np.median(lens))}, 99th pct {int(np.percentile(lens, 99))} | "
          f"truncated at {MAX_LEN}: {np.mean(np.array(lens) > MAX_LEN)*100:.1f}%")
    assert np.mean(np.array(lens) > MAX_LEN) < 0.02, "More than 2% of texts are cut off — raise MAX_LEN"

    # Training sample: every positive + NEG_PER_POS negatives each
    pos = train_df[train_df.respiratory_failure == 1]
    neg = train_df[train_df.respiratory_failure == 0].sample(min(len(train_df) - len(pos), NEG_PER_POS * len(pos)), random_state=SEED)
    sample = pd.concat([pos, neg]).sample(frac=1, random_state=SEED)
    print(f"Training sample: {len(sample):,} ({len(pos):,} positive)")

    train_loader = encode(sample, shuffle=True)
    val_loader, test_loader = encode(val_df, bs=64), encode(test_df, bs=64)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    w = torch.tensor([1.0, len(neg) / len(pos)], dtype=torch.float, device=device)   # balance the sampled classes
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * len(train_loader) * EPOCHS), len(train_loader) * EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    best, history, best_state = -1, [], None
    for epoch in range(EPOCHS):
        model.train(); t0, tot = time.time(), 0.0
        for ids, mask, y in train_loader:
            opt.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                loss = loss_fn(model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits.float(), y.to(device))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        pv = predict(model, val_loader)
        m = metrics(val_df.respiratory_failure.values, pv)
        history.append({"epoch": epoch + 1, "train_loss": round(tot / len(train_loader), 4), **{f"val_{k}": v for k, v in m.items()}})
        print(f"  epoch {epoch+1}: loss {tot/len(train_loader):.4f} | val AUROC {m['AUROC']:.4f} | val AUPRC {m['AUPRC']:.4f} "
              f"(chance {m['AUPRC_chance']:.4f}) | {time.time()-t0:.0f}s")
        if m["AUPRC"] > best:
            best, best_epoch = m["AUPRC"], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    p_bert_val, p_bert_test = predict(model, val_loader), predict(model, test_loader)
    if SAVE_MODEL:
        model.save_pretrained(str(d / "clinicalbert_best")); tokenizer.save_pretrained(str(d / "clinicalbert_best"))

    # Structured model on the same split (same settings as notebook 04), then a simple average of ranks
    feats = [c for c in train_df.columns if c not in {"stay_id", "subject_id", "respiratory_failure", "clinical_text"}]
    gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=SEED)
    gb.fit(train_df[feats], train_df.respiratory_failure)
    p_gb_test = gb.predict_proba(test_df[feats])[:, 1]
    p_avg_test = (rankdata(p_bert_test) + rankdata(p_gb_test)) / 2

    y = test_df.respiratory_failure.values
    res = {"best_epoch_by_val_AUPRC": best_epoch, "history": history,
           "test": {"ClinicalBERT": metrics(y, p_bert_test), "GradBoost (structured)": metrics(y, p_gb_test),
                    "Average of both": metrics(y, p_avg_test)},
           "n_train_sample": len(sample), "n_test": len(test_df)}
    print(pd.DataFrame(res["test"]).T.to_string())
    del model; torch.cuda.empty_cache()
    return res

In [6]:
all_res = {f"{L}h": run(L) for L in LANDMARKS}
out = SAVE_DIR / "clinicalbert_landmark_results.json"
with open(out, "w") as f:
    json.dump(all_res, f, indent=2)
print(f"\nSaved → {out}")


=== Landmark 12h === train 34,099 | val 7,366 | test 7,241 | positive rate 7.9%
Tokens per text: median 266, 99th pct 331 | truncated at 384: 0.0%
Training sample: 10,748 (2,687 positive)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

  epoch 1: loss 0.6683 | val AUROC 0.6813 | val AUPRC 0.1570 (chance 0.0832) | 258s
  epoch 2: loss 0.6454 | val AUROC 0.6958 | val AUPRC 0.1634 (chance 0.0832) | 254s
  epoch 3: loss 0.6349 | val AUROC 0.6944 | val AUPRC 0.1649 (chance 0.0832) | 254s
  epoch 4: loss 0.6263 | val AUROC 0.6934 | val AUPRC 0.1649 (chance 0.0832) | 255s
                         AUROC   AUPRC  AUPRC_chance
ClinicalBERT            0.6776  0.1488        0.0739
GradBoost (structured)  0.7158  0.1771        0.0739
Average of both         0.7084  0.1727        0.0739

Saved → /content/drive/MyDrive/USC/ICU-MM-landmark/clinicalbert_landmark_results.json
